In [5]:
import numpy as np


# Calculate fitness of a wolf
def fitness(solution, install_cost, operating_cost,
            demand, distances, service_distance, weights):

    selected = np.where(solution == 1)[0]

    if len(selected) == 0:
        return float('inf'), 0, 0, 0, 0

    total_install = np.sum(install_cost[selected])
    total_operating = np.sum(operating_cost[selected])

    # Find nearest selected station for each user
    nearest = np.min(distances[:, selected], axis=1)

    total_demand = np.sum(demand)

    # Demand-weighted average travel distance
    avg_distance = np.sum(nearest * demand) / total_demand

    # Calculate demand coverage
    covered = nearest <= service_distance

    coverage = (
        np.sum(demand[covered]) / total_demand
    ) * 100

    # Normalize objective components
    cost_scale = max(
        np.sum(install_cost + operating_cost), 1
    )

    distance_scale = max(np.max(distances), 1)

    normalized_cost = (
        total_install + total_operating
    ) / cost_scale

    normalized_distance = avg_distance / distance_scale

    uncovered = 1 - coverage / 100

    score = (
        weights[0] * normalized_cost +
        weights[1] * normalized_distance +
        weights[2] * uncovered
    )

    return (
        score, total_install, total_operating,
        avg_distance, coverage
    )


# Binary conversion using sigmoid function
def binary_conversion(position):

    position = np.clip(position, -20, 20)

    probability = 1 / (1 + np.exp(-position))

    return (np.random.random(len(position)) < probability).astype(int)


# Grey Wolf Optimizer
def gwo(install_cost, operating_cost, demand,
        distances, service_distance,
        population_size, max_iterations, weights):

    dimensions = len(install_cost)

    # Initialize wolf positions
    wolves = np.random.uniform(
        -2, 2, (population_size, dimensions)
    )

    alpha_position = None
    beta_position = None
    delta_position = None

    alpha_score = float('inf')
    beta_score = float('inf')
    delta_score = float('inf')

    # GWO iterations
    for iteration in range(max_iterations):

        # Evaluate all wolves
        for i in range(population_size):

            solution = binary_conversion(wolves[i])

            result = fitness(
                solution, install_cost,
                operating_cost, demand,
                distances, service_distance, weights
            )

            score = result[0]

            # Update Alpha, Beta and Delta
            if score < alpha_score:
                delta_score = beta_score
                delta_position = beta_position

                beta_score = alpha_score
                beta_position = alpha_position

                alpha_score = score
                alpha_position = wolves[i].copy()

            elif score < beta_score:
                delta_score = beta_score
                delta_position = beta_position

                beta_score = score
                beta_position = wolves[i].copy()

            elif score < delta_score:
                delta_score = score
                delta_position = wolves[i].copy()

        # Decrease exploration parameter
        a = 2 - 2 * iteration / max_iterations

        # Update wolf positions
        for i in range(population_size):

            for j in range(dimensions):

                # Alpha wolf
                r1, r2 = np.random.random(2)
                A1 = 2 * a * r1 - a
                C1 = 2 * r2

                D_alpha = abs(
                    C1 * alpha_position[j] - wolves[i][j]
                )

                X1 = alpha_position[j] - A1 * D_alpha

                # Beta wolf
                r1, r2 = np.random.random(2)
                A2 = 2 * a * r1 - a
                C2 = 2 * r2

                D_beta = abs(
                    C2 * beta_position[j] - wolves[i][j]
                )

                X2 = beta_position[j] - A2 * D_beta

                # Delta wolf
                r1, r2 = np.random.random(2)
                A3 = 2 * a * r1 - a
                C3 = 2 * r2

                D_delta = abs(
                    C3 * delta_position[j] - wolves[i][j]
                )

                X3 = delta_position[j] - A3 * D_delta

                # Update position
                wolves[i][j] = (X1 + X2 + X3) / 3

    # Convert best Alpha position to binary solution
    best_solution = binary_conversion(alpha_position)

    # Ensure at least one station is selected
    if np.sum(best_solution) == 0:
        best_solution[np.argmax(alpha_position)] = 1

    return best_solution


# Main program
def main():

    print("EV CHARGING STATION PLACEMENT")
    print("Using Grey Wolf Optimizer")

    n = int(input("\nEnter number of candidate locations: "))
    m = int(input("Enter number of EV users: "))

    if n <= 0 or m <= 0:
        print("Invalid number of locations or users.")
        return

    install_cost = np.array([
        float(input(f"Installation cost of location {i+1}: "))
        for i in range(n)
    ])

    operating_cost = np.array([
        float(input(f"Operating cost of location {i+1}: "))
        for i in range(n)
    ])

    demand = np.array([
        float(input(f"Charging demand of user {i+1}: "))
        for i in range(m)
    ])

    # Distance matrix
    distances = np.zeros((m, n))

    print("\nEnter distances from users to locations:")

    for i in range(m):
        for j in range(n):
            distances[i][j] = float(
                input(f"User {i+1} to Location {j+1}: ")
            )

    service_distance = float(
        input("\nEnter maximum service distance: ")
    )

    population_size = int(
        input("Enter number of wolves: ")
    )

    max_iterations = int(
        input("Enter maximum iterations: ")
    )

    print("\nEnter objective weights:")

    w1 = float(input("Installation and operating cost weight: "))
    w2 = float(input("Travel distance weight: "))
    w3 = float(input("Coverage weight: "))

    if (population_size < 3 or max_iterations <= 0 or
            np.any(install_cost < 0) or
            np.any(operating_cost < 0) or
            np.any(demand < 0) or
            np.any(distances < 0) or
            service_distance < 0 or
            np.sum(demand) <= 0 or
            min(w1, w2, w3) < 0 or
            w1 + w2 + w3 <= 0):

        print("Invalid input values.")
        return

    weights = np.array([w1, w2, w3])
    weights = weights / np.sum(weights)

    # Run GWO
    best_solution = gwo(
        install_cost, operating_cost,
        demand, distances, service_distance,
        population_size, max_iterations, weights
    )

    # Calculate final results
    result = fitness(
        best_solution, install_cost,
        operating_cost, demand,
        distances, service_distance, weights
    )

    print("\n------ OPTIMAL RESULT ------")

    selected = np.where(best_solution == 1)[0]

    print("\nSelected charging station locations:")

    for location in selected:
        print("Location", location + 1)

    print("\nTotal number of stations:", len(selected))
    print("Installation cost:", result[1])
    print("Operating cost:", result[2])
    print("Average travel distance:", round(result[3], 2))
    print("Demand coverage:", round(result[4], 2), "%")
    print("Best fitness value:", round(result[0], 5))


if __name__ == "__main__":
    main()


2

EV CHARGING STATION PLACEMENT
Using Grey Wolf Optimizer

Enter number of candidate locations: 3
Enter number of EV users: 4
Installation cost of location 1: 500000
Installation cost of location 2: 800000
Installation cost of location 3: 600000
Operating cost of location 1: 50000
Operating cost of location 2: 40000
Operating cost of location 3: 45000
Charging demand of user 1: 10
Charging demand of user 2: 20
Charging demand of user 3: 15
Charging demand of user 4: 25

Enter distances from users to locations:
User 1 to Location 1: 2
User 1 to Location 2: 8
User 1 to Location 3: 5
User 2 to Location 1: 7
User 2 to Location 2: 3
User 2 to Location 3: 6
User 3 to Location 1: 4
User 3 to Location 2: 6
User 3 to Location 3: 2
User 4 to Location 1: 9
User 4 to Location 2: 4
User 4 to Location 3: 7

Enter maximum service distance: 5
Enter number of wolves: 20
Enter maximum iterations: 100

Enter objective weights:
Installation and operating cost weight: 0.4
Travel distance weight: 0.3
Coverage